# Reference · Day 8 studio — the number you report

**Not a marking key.** Everyone worked on the same split and the same 300 houses, so if
you followed the studio your numbers should match these. If they do not, the interesting
question is which choice you made differently.

The studio asked for one habit: **report the number the search did not choose by.**

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

import warnings

import numpy as np
from stat764 import load

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", message="Skipping features without any observed values")

ames_all = load("ames.csv")
ames = ames_all[ames_all["Gr_Liv_Area"] <= 4000].reset_index(drop=True)   # as in the meeting
y = ames["SalePrice"]
WIDE_NUM = [c for c in ames.select_dtypes("number").columns if c not in ("Order", "PID", "SalePrice")]
WIDE_CAT = ames.select_dtypes(exclude="number").columns.tolist()
ALPHAS = np.logspace(-1, 4, 11)
LASSO_ALPHAS = [30, 100, 300, 1000, 3000]


def make_pipe(numeric, categorical, model=None):
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                              ("scale", StandardScaler())]), numeric),
            ("cat", Pipeline([("fill", SimpleImputer(strategy="constant", fill_value="Missing")),
                              ("encode", OneHotEncoder(handle_unknown="ignore"))]), categorical),
        ])),
        ("model", model or LinearRegression()),
    ])


X_train, X_test, y_train, y_test = train_test_split(
    ames[WIDE_NUM + WIDE_CAT], y, test_size=0.25, random_state=764)
X300, y300 = X_train.iloc[:300], y_train.iloc[:300]               # Tuesday's 300 houses

inner = KFold(5, shuffle=True, random_state=764)     # INNER folds: the search CHOOSES alpha
outer = KFold(5, shuffle=True, random_state=1764)    # OUTER folds: they GRADE the whole search

## The lasso, scored three ways

In [ ]:
# The search on the INNER folds: it chooses alpha, then refits on all 300 houses.
lasso_search = GridSearchCV(make_pipe(WIDE_NUM, WIDE_CAT, Lasso(max_iter=20000)),
                            {"model__alpha": LASSO_ALPHAS}, cv=inner, scoring="r2")
lasso_search.fit(X300, y300)
# Nested CV on the OUTER folds: the whole search, rerun and graded inside each one.
nested = cross_val_score(lasso_search, X300, y300, cv=outer, scoring="r2")
print(f"  the lasso chose alpha = {lasso_search.best_params_['model__alpha']:g}")
print(f"  best_score_  the score it chose by          {lasso_search.best_score_:.3f}")
print(f"  nested CV    the whole search, scored       {nested.mean():.3f}")
print(f"  held out     touched once, at the very end  {r2_score(y_test, lasso_search.predict(X_test)):.3f}")

### What this shows

**The sentence for the report:** *"Tuned inside 5-fold cross-validation, the lasso's
nested cross-validated R-squared on 300 houses is 0.875."* Not 0.902 — that is the score
the search chose α by, so it is pushed up by the choosing.

**Why is the held-out number (0.901) higher than nested?** One test set of 732 houses is
one draw, and nested is a touch pessimistic — each of its outer folds trains on 240
houses, not 300. Across ten other splits `best_score_` sat above the nested estimate on
**8 of 10**. On a small, smooth grid like this one the curse is small. You only know that
because you checked.

⚠ **The common slip:** `cross_val_score(lasso_search.best_estimator_, ...)`. That re-uses
the α the search already chose on *all* 300 houses, so it is the winner scored again —
not nested. Nested CV hands `cross_val_score` the **search**.

## How nested CV actually runs — the lasso, by hand

`cross_val_score(lasso_search, ...)` is one line, and it hides everything. Here it is
written out as three loops: the outer folds **grade**; the α loop and the inner folds
**choose**. Same folds as above, so it lands on the same number.

In [ ]:
grades = []
for out_tr, out_te in outer.split(X300):                       # LOOP 1: each outer fold
    X_o, y_o = X300.iloc[out_tr], y300.iloc[out_tr]            # the 240 the search may use
    cv_score = {}
    for a in LASSO_ALPHAS:                                     # LOOP 2: every alpha
        scores = []
        for in_tr, in_va in inner.split(X_o):                  # LOOP 3: inner folds, 192 / 48
            fit = make_pipe(WIDE_NUM, WIDE_CAT, Lasso(alpha=a, max_iter=20000)).fit(
                X_o.iloc[in_tr], y_o.iloc[in_tr])
            scores.append(r2_score(y_o.iloc[in_va], fit.predict(X_o.iloc[in_va])))
        cv_score[a] = np.mean(scores)                          # this alpha's CV score
    best = max(cv_score, key=cv_score.get)                     # the pick — a rehearsal
    final = make_pipe(WIDE_NUM, WIDE_CAT, Lasso(alpha=best, max_iter=20000)).fit(X_o, y_o)
    grades.append(r2_score(y300.iloc[out_te], final.predict(X300.iloc[out_te])))
    print(f"  outer fold {len(grades)}: picked alpha {best:g}, "
          f"graded {grades[-1]:.3f} on {len(out_te)} houses it never saw")
print(f"\n  nested score, by hand            {np.mean(grades):.3f}")
print(f"  cross_val_score(lasso_search)    {nested.mean():.3f}")

### What this shows

**The outer folds' picks are thrown away.** They exist only so the choosing can be
graded. They are still worth a look, as a **stability check**: the lasso picked α = 100,
1,000, 300, 100, 100 across the five folds, while ridge on the same folds picked 10, 10,
10, 10, 3.16 (Meeting 8, Part 3). Near the top the lasso's CV scores are almost tied, so
which α wins is close to a coin flip.

**So which α do you use?** The ordinary search on all the data: `lasso_search.fit(X300,
y300)` chose α = 300 and refit it on all 300 houses — that is the model. Nested CV never
chooses your final model; it gives you the honest number to report for it, 0.875.

**Watch it step by step:**
[the nested cross-validation animation, on the course site](https://datascienceuwl.github.io/STAT764/nested-cv/)
— Part 1 is the number to report, Part 2 the model to use.

## If you had time: peeking on purpose

In [ ]:
peeked = {a: r2_score(y_test, make_pipe(WIDE_NUM, WIDE_CAT, Ridge(alpha=a))
                      .fit(X300, y300).predict(X_test)) for a in ALPHAS}
best_a = max(peeked, key=peeked.get)
honest = GridSearchCV(make_pipe(WIDE_NUM, WIDE_CAT, Ridge()), {"model__alpha": ALPHAS},
                      cv=inner, scoring="r2").fit(X300, y300)
print(f"  choosing alpha by the TEST score picks alpha = {best_a:g}: {peeked[best_a]:.3f}")
print(f"  the honest search picked alpha = {honest.best_params_['model__alpha']:g}: "
      f"{r2_score(y_test, honest.predict(X_test)):.3f}")

### What this shows

**Peeking bought nothing here** — it picked the same α the honest search did. Across ten
splits it bought a median of +0.001, and at most +0.053.

**That is the danger, not the reassurance.** A report that peeked and a report that did
not look identical. Once the test set has helped choose, its number is no longer a
held-out number — and nobody reading the report can tell.

## Five questions to ask yourself

1. Did I report the number the search did **not** choose by?
2. Did I hand `cross_val_score` the **search**, not its best estimator?
3. Did I look at the test set **once**, after choosing — or did it help me choose?
4. Could a reader tell, from my report alone, how hard I searched?
5. If I had tried two hundred models instead of five values of α, would I still trust
   `best_score_`?